### 本文件用于探究lerobot mutidataset的使用方法

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1" 

#### 1. 本地加载单数据集

本地目录结构应为：

```
/vla/.data/
  test/            ← 一个 repo 一个子目录
  adjust_bottle/
  adjust_bottle3/
```

**推荐写法**（`root` 指向父目录 + 短 `repo_id`）：

```python
LeRobotDataset("test", root=Path("/vla/.data") / "test")
# 等价于 MultiLeRobotDataset 内部实际使用的路径
```

也支持绝对路径写法 `LeRobotDataset("/vla/.data/test")`，但 MultiLeRobotDataset 更推荐 `root + repo_id`。

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds1 = LeRobotDataset("/vla/.data/test")
ds1

LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '144',
    Number of selected samples: '22733',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_rgb', 'observation.images.robot0_agentview_right_rgb', 'observation.images.robot0_eye_in_hand_rgb', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds2 = LeRobotDataset("/vla/.data/adjust_bottle")
ds2

LeRobotDataset({
    Repository ID: '/vla/.data/adjust_bottle',
    Number of selected episodes: '50',
    Number of selected samples: '10548',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

#### 2. 多数据集加载（`MultiLeRobotDataset`）

官方要求子数据集 **feature 名 + shape 一致**（至少 `action` / `observation.state` 维度相同），否则在 `aggregate_stats()` 报错：

```
ValueError: all input arrays must have the same shape
```

| 组合 | 能否混合 | 原因 |
|------|---------|------|
| `adjust_bottle` + `adjust_bottle3` | ✅ | 同为 aloha，state/action 都是 14 维 |
| `test` + `adjust_bottle` | ❌ | state 9 vs 14，action 12 vs 14 |

+ 图像相关的 keys 必须完全相同（名称和类型）
+ state 和 action 需要维度相同
+ fps 最好一致

In [16]:
from lerobot.datasets.lerobot_dataset import MultiLeRobotDataset, LeRobotDatasetMetadata
multi_ds = MultiLeRobotDataset(
    repo_ids=["/vla/.data/adjust_bottle3", "/vla/.data/adjust_bottle"],
)
multi_ds

MultiLeRobotDataset(
  Repository IDs: '['/vla/.data/adjust_bottle3', '/vla/.data/adjust_bottle']',
  Number of Samples: 21096,
  Number of Episodes: 100,
  Type: image (.png),
  Recorded Frames per Second: 50,
  Camera Keys: ['observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist'],
  Video Frame Keys: N/A,
  Transformations: None,
)

In [17]:
from pathlib import Path
def check_multi_compatible(repo_ids: list[str], root: Path) -> None:
    """混合前检查：feature 交集 + action/state shape 是否一致。"""
    metas = [LeRobotDatasetMetadata(rid, root=root / rid) for rid in repo_ids]
    common = set(metas[0].features)
    for m in metas[1:]:
        common &= set(m.features)
    print("公共 feature keys:", sorted(common))
    for key in ["observation.state", "action"]:
        shapes = {rid: tuple(metas[i].features[key]["shape"]) for i, rid in enumerate(repo_ids)}
        ok = len(set(shapes.values())) == 1
        print(f"  {key} shapes: {shapes}  {'✅' if ok else '❌ 维度不一致，MultiLeRobotDataset 会失败'}")
LOCAL_ROOT = Path("/vla/.data")
check_multi_compatible(["test", "adjust_bottle"], LOCAL_ROOT)

# ❌ 不兼容：test(franka, 9/12维) + adjust_bottle(aloha, 14/14维)
try:
    multi_ds_mixed = MultiLeRobotDataset(
        repo_ids=["test", "adjust_bottle"],
        root=LOCAL_ROOT,
        download_videos=False,
    )
except ValueError as e:
    print("\n预期报错:", e)
    print("原因: aggregate_stats 要把各子数据集的 mean/std 做 stack，shape 不同就会 ValueError")

公共 feature keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.state', 'task_index', 'timestamp']
  observation.state shapes: {'test': (9,), 'adjust_bottle': (14,)}  ❌ 维度不一致，MultiLeRobotDataset 会失败
  action shapes: {'test': (12,), 'adjust_bottle': (14,)}  ❌ 维度不一致，MultiLeRobotDataset 会失败



预期报错: all input arrays must have the same shape
原因: aggregate_stats 要把各子数据集的 mean/std 做 stack，shape 不同就会 ValueError
